In [ ]:

import pandas as pd

# Load the datasets
train_data_path = 'D:/LLM-Driven_AI-Studio/MLAgent/data/benchmark/ml_benchmark/06_santander_customer/split_train.csv'
eval_data_path = 'D:/LLM-Driven_AI-Studio/MLAgent/data/benchmark/ml_benchmark/06_santander_customer/split_eval.csv'

train_df = pd.read_csv(train_data_path)
eval_df = pd.read_csv(eval_data_path)

# Display the first few rows of the datasets
print(train_df.head())
print(eval_df.head())


   target    var_0   var_1    var_2  ...   var_14   var_15   var_16   var_17
0       0  12.6407  2.9411  15.2365  ...  11.1089  15.0745   4.8570  -3.3979
1       0   5.5300 -2.6867  10.0739  ...   2.8759  13.9574   6.2063   6.6106
2       0   9.3848  1.3839   9.4459  ...  10.4819  14.2421   3.8085 -10.8337
3       0  11.5179 -5.7696  12.4163  ...   6.4419  14.8648   8.5745  -0.7653
4       0  12.6522 -6.0295  12.0836  ...   8.3703  14.4378  11.4569   0.4568

[5 rows x 19 columns]
   target    var_0   var_1    var_2  ...  var_14   var_15   var_16   var_17
0       1   8.6646 -2.0466  12.2794  ...  6.1990  14.3138  11.9914 -18.2099
1       1  18.2832 -0.0892  10.4312  ...  8.2536  14.2073  10.8528  -2.7252
2       0  13.8916 -0.3960   8.1494  ...  3.6420  14.9770   9.0804 -14.1675
3       0   8.2493 -0.0634  10.6337  ...  8.9594  14.5846   3.3459   2.4677
4       0   9.6026  2.7460   5.8710  ...  7.7771  13.9491  10.4395 -12.0996

[5 rows x 19 columns]


In [ ]:


# Check for missing values
print("Missing values in train:", train_df.isnull().sum().any())
print("Missing values in eval:", eval_df.isnull().sum().any())

# Get basic statistics
print(train_df.describe())
print(eval_df.describe())


Missing values in train: False
Missing values in eval: False
              target          var_0  ...         var_16         var_17
count  160000.000000  160000.000000  ...  160000.000000  160000.000000
mean        0.100175      10.677401  ...       9.337185      -5.700292
std         0.300234       3.039160  ...       2.558483       6.715098
min         0.000000       0.408400  ...       0.635100     -33.380200
25%         0.000000       8.452700  ...       7.455175     -10.485750
50%         0.000000      10.524750  ...       9.235850      -5.663200
75%         0.000000      12.759200  ...      11.064625      -0.810775
max         1.000000      20.315000  ...      17.950600      19.025900

[8 rows x 19 columns]
             target         var_0  ...        var_16        var_17
count  40000.000000  40000.000000  ...  40000.000000  40000.000000
mean       0.101750     10.689968  ...      9.317579     -5.682488
std        0.302323      3.043630  ...      2.553141      6.702726
min      

In [ ]:

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

# Split the datasets into features and target
X_train = train_df.drop('target', axis=1)
y_train = train_df['target']

X_eval = eval_df.drop('target', axis=1)
y_eval = eval_df['target']

# Standardize the features
scaler = StandardScaler()

# Create a pipeline for scaling
preprocessor = ColumnTransformer(
    transformers=[
        ('num', scaler, X_train.columns)
    ]
)

# Fit and transform the training data
X_train_scaled = preprocessor.fit_transform(X_train)

# Transform the evaluation data
X_eval_scaled = preprocessor.transform(X_eval)



In [ ]:



from sklearn.preprocessing import PolynomialFeatures

# Create polynomial features
poly = PolynomialFeatures(degree=2, include_bias=False)

# Fit and transform the scaled training data
X_train_poly = poly.fit_transform(X_train_scaled)

# Transform the scaled evaluation data
X_eval_poly = poly.transform(X_eval_scaled)

# Print the shape of the original and poly features
print("Original features shape:", X_train_scaled.shape)
print("Poly features shape:", X_train_poly.shape)




Original features shape: (160000, 18)
Poly features shape: (160000, 189)


In [ ]:


from sklearn.linear_model import LogisticRegression
from sklearn.feature_selection import RFE

# Initialize the logistic regression model
log_reg = LogisticRegression(max_iter=10000)

# Use RFE to select top features
rfe = RFE(estimator=log_reg, n_features_to_select=10)
rfe.fit(X_train_poly, y_train)

# Get the selected features
selected_features = X_train.columns[rfe.support_]
print("Selected features:", selected_features)

# Transform the training and evaluation data using the selected features
X_train_selected = rfe.transform(X_train_poly)
X_eval_selected = rfe.transform(X_eval_poly)



---------------------------------------------------------------------------
IndexError                                Traceback (most recent call last)
Cell In[6], line 24
     22 pass
     23 print('##active_line12##')
---> 24 selected_features = X_train.columns[rfe.support_]
     25 print('##active_line13##')
     26 print('Selected features:', selected_features)

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\pandas\core\indexes\base.py:5416, in Index.__getitem__(self, key)
   5407         if len(key) == 0 and len(key) != len(self):
   5408             warnings.warn(
   5409                 "Using a boolean indexer with length 0 on an Index with "
   5410                 "length greater than 0 is deprecated and will raise in a "
   (...)
   5413                 stacklevel=find_stack_level(),
   5414             )
-> 5416 result = getitem(key)
   5417 # Because we ruled out integer above, we always get an arraylike here
   5418 if result.ndim 

In [ ]:



# Map the selected features back to the original features
selected_poly_indices = [i for i, val in enumerate(rfe.support_) if val]
selected_feature_names = poly.get_feature_names_out(X_train.columns)
selected_original_features = [selected_feature_names[i] for i in selected_poly_indices if '(' not in selected_feature_names[i]]

print("Selected original features:", selected_original_features)

# Transform the training and evaluation data using the selected features
from sklearn.preprocessing import PolynomialFeatures

poly = PolynomialFeatures(degree=2, include_bias=False)
X_train_poly = poly.fit_transform(X_train)
X_eval_poly = poly.transform(X_eval)

# Select the indices corresponding to the original selected features
from sklearn.feature_selection import RFE
log_reg = LogisticRegression(max_iter=10000)
rfe = RFE(estimator=log_reg, n_features_to_select=10)
rfe.fit(X_train_poly, y_train)

# Get the selected features
selected_poly_indices = [i for i, val in enumerate(rfe.support_) if val]
selected_feature_names = poly.get_feature_names_out(X_train.columns)
selected_original_features = [selected_feature_names[i] for i in selected_poly_indices if '(' not in selected_feature_names[i]]

print("Selected original features:", selected_original_features)

# Transform the training and evaluation data using the selected features
X_train_selected = X_train_poly[:, selected_poly_indices]
X_eval_selected = X_eval_poly[:, selected_poly_indices]




Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

ual(x, self.x):
    342     self._update_x(x)
--> 343 self._update_fun()
    344 self._update_grad()
    345 return self.f, self.g

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\scipy\optimize\_differentiable_functions.py:294, in ScalarFunction._update_fun(self)
    292 def _update_fun(self):
    293     if not self.f_updated:
--> 294         fx = self._wrapped_fun(self.x)
    295         if fx < self._lowest_f:
    296             self._lowest_x = self.x

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\scipy\optimize\_differentiable_functions.py:20, in _wrapper_fun.<locals>.wrapped(x)
     16 ncalls[0] += 1
     17 # Send a copy because the user may overwrite it.
     18 # Overwriting results in undefined behaviour because
     19 # fun(self

In [ ]:




from sklearn.preprocessing import PolynomialFeatures

# Create polynomial features of degree 1
poly = PolynomialFeatures(degree=1, include_bias=False)

# Fit and transform the training data
X_train_poly = poly.fit_transform(X_train)

# Transform the evaluation data
X_eval_poly = poly.transform(X_eval)

# Initialize the logistic regression model
log_reg = LogisticRegression(max_iter=10000)

# Fit the model on the polynomial features
log_reg.fit(X_train_poly, y_train)

# Predict probabilities on the evaluation data
y_eval_pred_proba = log_reg.predict_proba(X_eval_poly)[:, 1]

# Calculate the AUC score
from sklearn.metrics import roc_auc_score
auc = roc_auc_score(y_eval, y_eval_pred_proba)

print("AUC score on eval data:", auc)




AUC score on eval data: 0.6484739983765868


In [ ]:



from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

# Define the Random Forest model
rf_model = RandomForestClassifier(random_state=42)

# Define the parameter grid
param_grid = {
    'n_estimators': [100, 200],
    'max_depth': [None, 10, 20],
    'min_samples_split': [2, 5],
    'min_samples_leaf': [1, 2]
}

# Initialize GridSearchCV
grid_search = GridSearchCV(estimator=rf_model, param_grid=param_grid, cv=3, scoring='roc_auc', n_jobs=-1)

# Fit GridSearchCV
grid_search.fit(X_train_poly, y_train)

# Get the best model and its AUC score
best_model = grid_search.best_estimator_
y_eval_pred_proba_best = best_model.predict_proba(X_eval_poly)[:, 1]
auc_best = roc_auc_score(y_eval, y_eval_pred_proba_best)

print("Best model:", best_model)
print("AUC score on eval data with best model:", auc_best)





Output truncated. Showing the last 2800 characters. You should try again and use computer.ai.summarize(output) over the output, or break it down into smaller steps.

    with self._submit_resize_lock:
--> 225         return super().submit(fn, *args, **kwargs)

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\joblib\externals\loky\process_executor.py:1248, in ProcessPoolExecutor.submit(self, fn, *args, **kwargs)
   1245 # Wake up queue management thread
   1246 self._executor_manager_thread_wakeup.wakeup()
-> 1248 self._ensure_executor_running()
   1249 return f

File D:\LLM-Driven_AI-Studio\MLAgent\experiments\OpenInterpreter\venv\lib\site-packages\joblib\externals\loky\process_executor.py:1220, in ProcessPoolExecutor._ensure_executor_running(self)
   1218 with self._processes_management_lock:
   1219     if len(self._processes) != self._max_workers:
-> 1220         self._adjust_process_count()
   1221     self._start_executor_manager_thread()

Fi